In [3]:
"""
Project mortality effects for forecast ensemble and baseline period
"""

import datetime
import os
import uuid

import isku
import numpy as np
import xarray as xr
from dotenv import load_dotenv

from poreallas.extract import make_climtas, make_tas_monthly_histogram
from poreallas.project import calculate_beta, mortality_effect_model

In [ ]:
load_dotenv()
DATA_DIR = os.environ["DATA_DIR"]

# Input
# "Forecast" = Recent ERA5 2026 data (June-Present) + Historical ERA5 data (2006-2025)
RT_PERIOD = slice("2026-06-01", None)
TAS_RUNNING_URI = f"{DATA_DIR}/2609_era5_running_adj.zarr"
# "Baseline" = Historical ERA5 data (1996-2025)
ERA5_URI = os.environ["POREALLAS_ERA5_URI"]
# Response Function
GAMMA_URI = os.environ["POREALLAS_GAMMA_URI"]

SOCIOECONOMICS_URI = os.environ["POREALLAS_SOCIOECONOMICS_URI"]
# Impact Region Shapefile
REGIONS_URI = os.environ["POREALLAS_REGIONS_URI"]

# Output
EFFECTS_URI_RT = "v2609_effects_running_total.zarr"

In [5]:
def trim_ragged_months(
    ds: xr.Dataset, *, required_proportion: float = 0.9, datetime_dim="time"
) -> xr.Dataset:
    """
    Drop months without required number of days, returning a copy of input dataset.

    Forecast ensemble is for a fixed number of days so we expect to usually
    trim off the last month of the forecast if it is ragged and missing days beyond
    a threshold.
    """
    ds = ds.copy()

    n_initial = ds[datetime_dim].size
    number_obs = ds[datetime_dim].resample(time="ME").count()
    days_in_month = number_obs[datetime_dim].dt.days_in_month
    min_req = np.round(days_in_month * required_proportion)
    qualifying_months = number_obs.where(number_obs >= min_req, drop=True)[
        datetime_dim
    ].dt.month
    ds = ds.where(ds[datetime_dim].dt.month.isin(qualifying_months), drop=True)

    n_current = ds[datetime_dim].size
    n_initial_months = number_obs[datetime_dim].size
    n_qualifying_months = qualifying_months[datetime_dim].size

    print(
        f"continuing with {n_qualifying_months} of {n_initial_months} forecast months after removing incomplete months"
    )
    print(
        f"continuing with {n_current} of {n_initial} forecast periods after removing incomplete months"
    )

    assert (n_qualifying_months - n_initial_months) < 2, (
        "More than one incomplete month was removed from the forecast while checking for incomplete months. Something unexpected is happening."
    )
    return ds


def read_regions(uri: str) -> isku.GridWeightingRegions:
    _region_weights = xr.load_dataset(uri)[
        ["lat", "lon", "region", "weight"]
    ]  # Load only what we need.
    # Apparently in this version of xarray the `.load()` method type-hints it'll return a DataArray instead of a Dataset.
    # It is a Dataset (I checked). So telling ty to ignore it.
    # # TODO: send bug upstream?
    regions = isku.GridWeightingRegions(_region_weights)  # ty: ignore[invalid-argument-type]
    return regions

In [ ]:
# Read Data
reanalysis = xr.open_dataset(
    ERA5_URI, engine="zarr", chunks={"time": -1, "lat": 46, "lon": 92}
)  # For Climtas

running_reanalysis = xr.open_dataset(TAS_RUNNING_URI, engine="zarr", chunks={})

regions = read_regions(REGIONS_URI)

socioeconomics = xr.open_dataset(SOCIOECONOMICS_URI, engine="zarr", chunks={})

gammas = xr.open_dataset(GAMMA_URI, engine="zarr", chunks={})

# Select Only Running Total Months
running_reanalysis = running_reanalysis.sel(time=RT_PERIOD)
# Last months of forecast often missing significant number of days. Remove these ragged months.
running_reanalysis = trim_ragged_months(running_reanalysis)

/tmp/ipykernel_800/3369155868.py:40: RuntimeWarning: Failed to open Zarr store with consolidated metadata, but successfully read with non-consolidated metadata. This is typically much slower for opening a dataset. To silence this warning, consider:
1. Consolidating metadata in this existing store with zarr.consolidate_metadata().
2. Explicitly setting consolidated=False, to avoid trying to read consolidate metadata, or
3. Explicitly setting consolidated=True, to raise an error in this case instead of falling back to try reading non-consolidated metadata.
  _region_weights = xr.load_dataset(uri)[
/home/emily_zuetell/projects/poreallas/.venv/lib/python3.14/site-packages/zarr/core/group.py:3413: ZarrUserWarning: Object at zarr.json:Zone.Identifier is not recognized as a component of a Zarr hierarchy.
  warnings.warn(


continuing with 237 of 237 forecast months after removing incomplete months
continuing with 7208 of 7208 forecast periods after removing incomplete months
continuing with 4 of 4 forecast months after removing incomplete months
continuing with 122 of 122 forecast periods after removing incomplete months


In [ ]:
# Transform gridded data, extracting regional data needed for projections.
histogram_hist_tas = isku.extract_regions(
    reanalysis,
    template=make_tas_monthly_histogram,
    regions=regions,
)
histogram_forecast_tas = isku.extract_regions(
    running_reanalysis,
    template=make_tas_monthly_histogram,
    regions=regions,
)

In [8]:
# Using the same static beta for forecast and reanalysis projection requires the histogram tas_bin for these data need to be equal, too. So we're  calculating it here.
xr.testing.assert_allclose(
    histogram_hist_tas["tas_bin"],
    histogram_forecast_tas["tas_bin"],
)

climtas = isku.extract_regions(
    reanalysis,
    template=make_climtas,
    regions=regions,
).sel(year=2025, drop=True)

# log(GDPpc + 1), the + 1 prevents warning about undefined behavior when log is fed a GDPpc of 0.
loggdppc = np.log(socioeconomics["gdppc"].sel(year=2023, drop=True) + 1)

In [9]:
# Calculate a fixed response function, i.e. beta.
# Single, static response function with no adaptation is used for both projections.
# Stick everything together and make sure it aligns and matches. Rechunk all together. Also drop any regions with NaNs.
beta_input = (
    xr.Dataset(
        {
            "tas_bin": histogram_forecast_tas["tas_bin"],
            "climtas": climtas["climtas"],
            "loggdppc": loggdppc,
            "gamma": gammas["gamma_sampled"],
        }
    )
    .dropna(dim="region")
    .chunk(
        {
            "region": "auto",  # "auto" is a sensible default.
            "tas_bin": -1,  # This also needs to be all in memory.
            "age_cohort": 1,  # We're doing all age_cohorts at once but could be done one-by-one.
            "degree": -1,  # For gammas and polynomial calculations. Should all be in memory.
            "sample": 1,  # Dimension for gamma draws. Having them in memory or not is not strictly required.
        },
    )
    .unify_chunks()
)
fixed_beta = calculate_beta(beta_input).astype("float32").compute()
fixed_beta["beta"].attrs = {
    "units": "deaths per 100,000 people",
    "long_name": "Temperature mortality rate",
}
# Do beta allowing only hot deaths by 0-ing out everthing on the cold side of the minimum-mortality temperature.
fixed_beta["beta_hotonly"] = fixed_beta["beta"].where(
    fixed_beta["tas_bin"] > fixed_beta["mmt"], other=0
)
fixed_beta["beta_hotonly"].attrs["long_name"] = "Hot temperature mortality rate"

# Do beta, allowing only COLD deaths by 0-ing out everything on the warm side of the minimum-mortality temperature.
fixed_beta["beta_coldonly"] = fixed_beta["beta"].where(
    fixed_beta["tas_bin"] < fixed_beta["mmt"], other=0
)
fixed_beta["beta_coldonly"].attrs["long_name"] = "Cold temperature mortality rate"

In [11]:
# Project mortality.
# Start with forecast ensemble.
forecast_input = (
    xr.Dataset(
        {
            "histogram_tas": histogram_forecast_tas["histogram_tas"],
            "beta": fixed_beta["beta"],
        }
    )
    .dropna(dim="region")
    .chunk(
        {
            "region": "auto",  # "auto" is a sensible default.
            "time": -1,
            "tas_bin": -1,
            "age_cohort": 1,
            "sample": 1,
        },
    )
    .unify_chunks()
)
projected_forecast = isku.project(forecast_input, model=mortality_effect_model)
projected_forecast["effect"].attrs = {
    "units": "deaths per 100,000 people",
    "long_name": "Temperature mortality",
}

# Now hot-only projection
forecast_input = (
    xr.Dataset(
        {
            "histogram_tas": histogram_forecast_tas["histogram_tas"],
            "beta": fixed_beta["beta_hotonly"],
        }
    )
    .dropna(dim="region")
    .chunk(
        {
            "region": "auto",  # "auto" is a sensible default.
            "time": -1,
            "tas_bin": -1,
            "age_cohort": 1,
            "sample": 1,
        },
    )
    .unify_chunks()
)
projected_forecast_hotonly = isku.project(forecast_input, model=mortality_effect_model)
projected_forecast_hotonly["effect"].attrs = {
    "units": "deaths per 100,000 people",
    "long_name": "Hot temperature mortality",
}

In [ ]:
# Collect everything and write to storage.
_out = {
    "forecast_runningtotal": projected_forecast,
    "forecast_hotonly_runningtotal": projected_forecast_hotonly,
    "fixed_beta": fixed_beta,
}
_out_dt = xr.DataTree.from_dict(_out)

# Add metadata
_uid = str(uuid.uuid4())
_datetime_now = datetime.datetime.now(datetime.UTC).isoformat()

_out_dt.attrs |= {
    "poreallas_created_at": _datetime_now,
    "poreallas_uid": _uid,
    "poreallas_description": "Projected temperature mortality effects",
}

_out_dt["forecast_runningtotal"].attrs |= {
    "poreallas_created_at": _datetime_now,
    "poreallas_uid": _uid,
    "poreallas_description": "Forecast ensemble projected temperature mortality effects",
    "poreallas_temperature_uri": TAS_RUNNING_URI,
    "poreallas_socioeconomics_uri": SOCIOECONOMICS_URI,
    "poreallas_model_parameters_uri": GAMMA_URI,
    "poreallas_regions_uri": REGIONS_URI,
}

_out_dt["forecast_hotonly_runningtotal"].attrs |= {
    "poreallas_created_at": _datetime_now,
    "poreallas_uid": _uid,
    "poreallas_description": "Forecast ensemble projected hot temperature mortality effects",
    "poreallas_temperature_uri": TAS_RUNNING_URI,
    "poreallas_socioeconomics_uri": SOCIOECONOMICS_URI,
    "poreallas_model_parameters_uri": GAMMA_URI,
    "poreallas_regions_uri": REGIONS_URI,
}

_out_dt["fixed_beta"].attrs |= {
    "poreallas_created_at": _datetime_now,
    "poreallas_uid": _uid,
    "poreallas_description": "Fixed betas or temperature-mortality rates used in projections",
    "poreallas_temperature_uri": ERA5_URI,
    "poreallas_socioeconomics_uri": SOCIOECONOMICS_URI,
    "poreallas_model_parameters_uri": GAMMA_URI,
    "poreallas_regions_uri": REGIONS_URI,
}

if EFFECTS_URI_RT is not None:
    _out_dt.to_zarr(EFFECTS_URI_RT, consolidated=True)
    print(f"Effects written to {EFFECTS_URI_RT}")

Effects written to 2609_effects_running_total_tas.zarr
